# Oseen Flow Around a Cylinder with Inlet Total Pressure

This notebook documents the Julia script that solves the Oseen linearization of cylinder flow with a total-pressure condition at the inlet. The inlet condition is explained explicitly in the math so the boundary treatment is clear and easy to audit.

## 1. Governing equations

The Oseen system is the linearized form of incompressible Navier-Stokes obtained by freezing the advecting velocity to a prescribed reference field $\mathbf{w}$. The strong form is

$$
\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{w} \cdot \nabla)\mathbf{u} - \nu \nabla^2 \mathbf{u} + \nabla p = \mathbf{f},
$$

$$
\nabla \cdot \mathbf{u} = 0.
$$

Here $\mathbf{w}$ is a known reference velocity, $\mathbf{u}$ is the unknown velocity, and $p$ is pressure. This linearization keeps the transport effect but removes the quadratic nonlinearity of Navier-Stokes.

## 2. Weak form and Oseen operator

Testing with $\mathbf{v}$ and $q$ gives

$$
\int_\Omega \frac{\partial \mathbf{u}}{\partial t} \cdot \mathbf{v}\, d\Omega
+ \int_\Omega [ (\mathbf{w} \cdot \nabla)\mathbf{u} ] \cdot \mathbf{v}\, d\Omega
+ \nu \int_\Omega \nabla \mathbf{u} : \nabla \mathbf{v}\, d\Omega
- \int_\Omega p \nabla \cdot \mathbf{v}\, d\Omega = 0,
$$

$$
\int_\Omega q \nabla \cdot \mathbf{u}\, d\Omega = 0.
$$

After spatial discretization this becomes a linear DAE in the unknown coefficients. The convective matrix uses the fixed reference velocity $\mathbf{w}$, not the unknown field.

In [ ]:
using Ferrite, FerriteGmsh, Tensors, LinearAlgebra, SparseArrays, WriteVTK
using FerriteGmsh: Gmsh

const nu = 1.0 / 1000.0
const rho = 2.0
const p0_inlet = 1.0
const w_oseen = Vec((0.1, 0.0))
const alpha_bc = 100.0
const H_channel = 0.41
const tol_corner = 0.0011
const newton_tol = 1e-8
const newton_maxiter = 30

## 3. Mesh and finite element spaces

The domain is again a channel with a circular obstacle. The script uses quadratic velocity and linear pressure on quadrilateral cells, which is the Taylor-Hood pair. The inlet, outlet, walls, and cylinder are tagged as separate boundary sets.

In [ ]:
function setup_grid(h = 0.05)
    Gmsh.initialize()
    gmsh.option.set_number("General.Verbosity", 2)
    dim = 2
    rect_tag = gmsh.model.occ.add_rectangle(0, 0, 0, 1.1, 0.41)
    circle_tag = gmsh.model.occ.add_circle(0.2, 0.2, 0, 0.05)
    circle_curve_tag = gmsh.model.occ.add_curve_loop([circle_tag])
    circle_surf_tag = gmsh.model.occ.add_plane_surface([circle_curve_tag])
    gmsh.model.occ.cut([(dim, rect_tag)], [(dim, circle_surf_tag)])
    gmsh.model.occ.synchronize()
    gmsh.model.model.add_physical_group(dim - 1, [6], -1, "bottom")
    gmsh.model.model.add_physical_group(dim - 1, [7], -1, "left")
    gmsh.model.model.add_physical_group(dim - 1, [8], -1, "right")
    gmsh.model.model.add_physical_group(dim - 1, [9], -1, "top")
    gmsh.model.model.add_physical_group(dim - 1, [5], -1, "hole")
    gmsh.option.setNumber("Mesh.Algorithm", 11)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 20)
    gmsh.option.setNumber("Mesh.MeshSizeMax", h)
    gmsh.model.mesh.generate(dim)
    grid = togrid()
    Gmsh.finalize()
    return grid
end

function setup_fevalues(ipv, ipp, ipg)
    qr = QuadratureRule{RefQuadrilateral}(4)
    cvv = CellValues(qr, ipv, ipg)
    cvp = CellValues(qr, ipp, ipg)
    qr_fac = FacetQuadratureRule{RefQuadrilateral}(4)
    fvv = FacetValues(qr_fac, ipv, ipg)
    fvp = FacetValues(qr_fac, ipp, ipg)
    return cvv, cvp, fvv, fvp
end

function setup_dofs(grid, ipv, ipp)
    dh = DofHandler(grid)
    add!(dh, :v, ipv)
    add!(dh, :p, ipp)
    close!(dh)
    return dh
end

## 4. Boundary conditions

The walls and cylinder are no-slip: $\mathbf{u}=\mathbf{0}$. The outlet pressure is fixed as a gauge to eliminate the pressure null-space. The inlet is again not given a velocity Dirichlet value. Instead, the same total-pressure relation is enforced weakly:

$$
p + \frac{1}{2}\rho u_x^2 = p_0,$$


In [ ]:
ch = ConstraintHandler(dh)

noslip_facet_names = ["top", "bottom", "hole"]
noslip_facets = union(getfacetset.((grid,), noslip_facet_names)...)
add!(ch, Dirichlet(:v, noslip_facets, (x, t) -> Vec((0.0, 0.0)), [1, 2]))

# Inlet has no Dirichlet velocity; it is driven by total-pressure penalty.
inlet_facets = getfacetset(grid, "left")

# Outlet pressure gauge to remove pressure null-space.
add!(ch, Dirichlet(:p, getfacetset(grid, "right"), (x, t) -> 0.0))

close!(ch)
update!(ch, 0.0)


## 5. Oseen linear operator

The Oseen stiffness matrix contains the viscous term and the linearized convection term with frozen velocity $\mathbf{w}$. In weak form the convective contribution is

$$
\int_\Omega [ (\mathbf{w} \cdot \nabla)\mathbf{u} ] \cdot \mathbf{v}\, d\Omega.
$$

This makes the interior operator linear, even though the inlet total-pressure condition remains nonlinear.

In [ ]:
function assemble_oseen_linear!(K, dh, cvv, cvp, nu, w)
    assembler = start_assemble(K)
    ke = zeros(ndofs_per_cell(dh), ndofs_per_cell(dh))
    range_v = dof_range(dh, :v)
    range_p = dof_range(dh, :p)
    ndofs_v = length(range_v)
    ndofs_p = length(range_p)
    phi_v = Vector{Vec{2, Float64}}(undef, ndofs_v)
    grad_phi_v = Vector{Tensor{2, 2, Float64, 4}}(undef, ndofs_v)
    div_phi_v = Vector{Float64}(undef, ndofs_v)
    phi_p = Vector{Float64}(undef, ndofs_p)
    for cell in CellIterator(dh)
        Ferrite.reinit!(cvv, cell)
        Ferrite.reinit!(cvp, cell)
        ke .= 0.0
        for qp in 1:getnquadpoints(cvv)
            dOmega = getdetJdV(cvv, qp)
            for i in 1:ndofs_v
                phi_v[i] = shape_value(cvv, qp, i)
                grad_phi_v[i] = shape_gradient(cvv, qp, i)
                div_phi_v[i] = shape_divergence(cvv, qp, i)
            end
            for i in 1:ndofs_p
                phi_p[i] = shape_value(cvp, qp, i)
            end
            for (i, I) in pairs(range_v), (j, J) in pairs(range_v)
                ke[I, J] += (nu * (grad_phi_v[i] ⊡ grad_phi_v[j]) + phi_v[i] ⋅ (grad_phi_v[j] ⋅ w)) * dOmega
            end
            for (i, I) in pairs(range_v), (j, J) in pairs(range_p)
                ke[I, J] += (-div_phi_v[i] * phi_p[j]) * dOmega
            end
            for (i, I) in pairs(range_p), (j, J) in pairs(range_v)
                ke[I, J] += (-phi_p[i] * div_phi_v[j]) * dOmega
            end
        end
        assemble!(assembler, celldofs(cell), ke)
    end
    return K
end

## 6. Inlet total-pressure equation

At the inlet we impose a scalar condition on the x-component of the velocity and the pressure. The physical statement is total pressure conservation,

$$
p_0 = p + \frac{1}{2}\rho u_x^2.
$$

This is rewritten as the residual

$$
r_{tp}(u_x,p) = u_x^2 - \frac{2}{\rho}(p_0 - p),
$$

and the residual is integrated on the inlet boundary with a penalty weight $\alpha_{bc}$. Its derivatives are

$$
\frac{\partial r_{tp}}{\partial u_x} = 2u_x, \qquad \frac{\partial r_{tp}}{\partial p} = \frac{2}{\rho}.
$$

These are exactly the terms used in the Jacobian assembly.

In [ ]:
function add_inlet_total_pressure_residual!(R, u, dh, fvv, fvp, inlet_boundary)
    v_range = dof_range(dh, :v)
    p_range = dof_range(dh, :p)
    for facet in FacetIterator(dh, inlet_boundary)
        Ferrite.reinit!(fvv, facet)
        Ferrite.reinit!(fvp, facet)
        celld = celldofs(facet)
        v_dofs = @view celld[v_range]
        p_dofs = @view celld[p_range]
        v_e = u[v_dofs]
        p_e = u[p_dofs]
        Re = zeros(length(v_dofs))
        for qp in 1:getnquadpoints(fvv)
            x = spatial_coordinate(fvv, qp, getcoordinates(facet))
            if x[2] <= tol_corner || x[2] >= H_channel - tol_corner
                continue
            end
            dGamma = getdetJdV(fvv, qp)
            ux = function_value(fvv, qp, v_e)[1]
            p_static = function_value(fvp, qp, p_e)
            res = total_p_residual(ux, p_static)
            for i in 1:getnbasefunctions(fvv)
                phi_x_i = shape_value(fvv, qp, i)[1]
                Re[i] -= alpha_bc * res * phi_x_i * dGamma
            end
        end
        assemble!(R, v_dofs, Re)
    end
end

function add_inlet_total_pressure_jacobian!(J, u, dh, fvv, fvp, inlet_boundary)
    v_range = dof_range(dh, :v)
    p_range = dof_range(dh, :p)
    assembler = start_assemble(J; fillzero = false)
    for facet in FacetIterator(dh, inlet_boundary)
        Ferrite.reinit!(fvv, facet)
        Ferrite.reinit!(fvp, facet)
        celld = celldofs(facet)
        v_dofs = @view celld[v_range]
        p_dofs = @view celld[p_range]
        v_e = u[v_dofs]
        Jvv = zeros(length(v_dofs), length(v_dofs))
        for qp in 1:getnquadpoints(fvv)
            x = spatial_coordinate(fvv, qp, getcoordinates(facet))
            if x[2] <= tol_corner || x[2] >= H_channel - tol_corner
                continue
            end
            dGamma = getdetJdV(fvv, qp)
            ux = function_value(fvv, qp, v_e)[1]
            dres_dux = dresidual_dux(ux)
            dres_dp = dresidual_dp()
            for i in 1:getnbasefunctions(fvv)
                phi_x_i = shape_value(fvv, qp, i)[1]
                for j in 1:getnbasefunctions(fvv)
                    phi_x_j = shape_value(fvv, qp, j)[1]
                    Jvv[i, j] -= alpha_bc * dres_dux * phi_x_i * phi_x_j * dGamma
                end
                for j in 1:getnbasefunctions(fvp)
                    psi_j = shape_value(fvp, qp, j)
                    J[v_dofs[i], p_dofs[j]] -= alpha_bc * dres_dp * phi_x_i * psi_j * dGamma
                end
            end
        end
        assemble!(assembler, v_dofs, Jvv)
    end
end

## 7. Nonlinear solve and output

The interior Oseen operator is linear, but the inlet total-pressure condition is nonlinear because of the $u_x^2$ term. The script therefore solves $F(u)=0$ with Newton's method, reapplying the constraints at each iteration. After convergence, the solution is written to VTK for ParaView. The final code cell also includes the original source file so the notebook can be executed end to end in this repository.

In [ ]:
function solve_oseen_total_pressure(dh, ch, K_lin, fvv, fvp, inlet_boundary)
    u = zeros(ndofs(dh))
    apply!(u, ch)
    R = zeros(ndofs(dh))
    for it in 1:newton_maxiter
        mul!(R, K_lin, u)
        add_inlet_total_pressure_residual!(R, u, dh, fvv, fvp, inlet_boundary)
        J = copy(K_lin)
        add_inlet_total_pressure_jacobian!(J, u, dh, fvv, fvp, inlet_boundary)
        apply!(J, R, ch)
        res_norm = norm(R)
        println("Newton iteration $it: ||R|| = $(res_norm)")
        if res_norm < newton_tol
            println("Newton converged.")
            return u
        end
        du = -(J \ R)
        u .+= du
        apply!(u, ch)
    end
    error("Newton did not converge in $(newton_maxiter) iterations.")
end

function main()
    grid = setup_grid(0.05)
    ipv = Lagrange{RefQuadrilateral, 2}()^2
    ipp = Lagrange{RefQuadrilateral, 1}()
    ipg = Lagrange{RefQuadrilateral, 1}()
    dh = setup_dofs(grid, ipv, ipp)
    cvv, cvp, fvv, fvp = setup_fevalues(ipv, ipp, ipg)
    ch = setup_constraints(dh)
    inlet_boundary = getfacetset(grid, "left")
    K_lin = allocate_matrix(dh, ch; coupling = [true true; true false])
    assemble_oseen_linear!(K_lin, dh, cvv, cvp, nu, w_oseen)
    u = solve_oseen_total_pressure(dh, ch, K_lin, fvv, fvp, inlet_boundary)
    VTKGridFile("oseen-p0-inlet-cyl", grid) do vtk
        write_solution(vtk, dh, u)
    end
    println("Done! Solution written to oseen-p0-inlet-cyl.vtu")
end
